# DriveAwareAI: Driver Distraction & Drowsiness Detection (Kaggle notebook)

This notebook is the Kaggle version of the **DriveAwareAI** project. It contains the project's pipeline as notebook cells, so it needs **no extra files**: attach the two datasets, enable a GPU, and **Run All**.

| Component | Implementation (same as the project) |
|---|---|
| Distraction model | ResNet-18 (ImageNet) → `Dropout → Linear(512, 10)`, 224 px, State Farm |
| Eye-state model | ResNet-18 (ImageNet) → `Dropout → Linear(512, 2)`, 160 px, MRL Eye |
| Temporal model | LayerNorm → LSTM(64) → additive temporal attention → `Linear(64, 3)` over fused per-frame vectors `[p(eye closed), p(c0..c9)]` |
| Decision layer | PERCLOS-style rules + LSTM blend (`fire if score ≥ 1 − w/2`) |
| Explainability | Grad-CAM on `layer4[-1]` (implemented here with hooks, no extra package) |

### How to use
1. **Add Input**, then attach:
   * **State Farm Distracted Driver Detection** (competition; click *Join Competition* first),
   * an **MRL Eye** dataset with `awake` / `sleepy` folders (e.g. `akashshingha850/mrl-eye-dataset`).
2. **Settings → Accelerator → GPU T4 ×2 or P100**. Internet should be **On** so the ImageNet weights can download.
3. **Run All**. All outputs go to `/kaggle/working/DriveAwareAI_results/`, and a ZIP with a download link is created at the end.

> **Resuming.** Training saves a checkpoint after every epoch. If a run is interrupted, re-run the cells: each training stage resumes from its last epoch. To resume in a *new* session, attach the previous run's output (which contains `DriveAwareAI_results/checkpoints`) as an input. The notebook finds and restores it.

> **Honesty note.** Neither dataset contains ordered video or drowsiness labels. The temporal model is therefore trained and tested on **simulated** sequences of encoder outputs (Section 12). Those numbers are **not** real-world drowsiness-detection performance.

*This notebook has been executed end to end on miniature synthetic datasets that mimic the Kaggle folder layout. It has not yet been run on Kaggle with the real datasets, so the first real run is the actual test.*

## 1. Environment setup and dependencies

The Kaggle Python image already includes everything this notebook needs: PyTorch, torchvision, OpenCV, NumPy, pandas, scikit-learn, matplotlib, seaborn and Pillow. Nothing is installed. The cell below only checks that each package imports and fails with a clear message if one is missing.

In [ ]:
import importlib, sys, platform

REQUIRED = ["torch", "torchvision", "numpy", "pandas", "sklearn", "matplotlib", "seaborn", "PIL", "cv2", "yaml"]
missing = []
versions = {"python": sys.version.split()[0], "platform": platform.platform()}
for name in REQUIRED:
    try:
        mod = importlib.import_module(name)
        versions[name] = getattr(mod, "__version__", "n/a")
    except ImportError:
        missing.append(name)
if missing:
    raise ImportError(
        f"Missing packages: {missing}. On Kaggle these are preinstalled; if you changed the "
        f"environment, install them with: pip install {' '.join(missing).replace('sklearn','scikit-learn').replace('PIL','pillow').replace('cv2','opencv-python-headless').replace('yaml','pyyaml')}")
for k, v in versions.items():
    print(f"{k:12s} {v}")

In [ ]:
import os, re, json, time, copy, math, random, shutil, zipfile, warnings
from pathlib import Path
from collections import Counter, deque

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import yaml
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torchvision import models, transforms

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, classification_report,
                             confusion_matrix, roc_auc_score)

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.width", 160)

## 2. Configuration, reproducibility and GPU detection

The settings are taken from the project's `configs/config.yaml`. `QUICK_RUN = True` gives a short smoke test (few epochs, data subsets). Leave it `False` for the real experiment.

The input and output roots default to Kaggle's `/kaggle/input` and `/kaggle/working`. You can override them with the environment variables `DRIVEAWARE_INPUT` and `DRIVEAWARE_WORKING` (used for local testing).

In [ ]:
QUICK_RUN = os.environ.get("DRIVEAWARE_QUICK", "0") == "1"   # set True for a fast smoke test

INPUT_ROOT = Path(os.environ.get("DRIVEAWARE_INPUT", "/kaggle/input"))
WORKING = Path(os.environ.get("DRIVEAWARE_WORKING", "/kaggle/working"))
RESULTS = WORKING / "DriveAwareAI_results"
DIRS = {k: RESULTS / k for k in ("checkpoints", "models", "plots", "reports", "gradcam", "history")}
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

CFG = {
    "seed": 42,
    "distraction": {
        "image_size": 224, "batch_size": 64, "epochs": 15, "lr_backbone": 1e-4, "lr_head": 3e-4,
        "weight_decay": 1e-4, "label_smoothing": 0.1, "dropout": 0.4,
        "freeze_modes": ["frozen", "finetune_layer3_4"],   # frozen-backbone vs fine-tuned comparison
        "val_fraction": 0.15, "test_fraction": 0.15, "early_stopping_patience": 4,
    },
    "eye_state": {
        "image_size": 160, "batch_size": 128, "epochs": 6, "lr": 1e-4, "weight_decay": 1e-4,
        "dropout": 0.3, "freeze_mode": "finetune_layer4", "early_stopping_patience": 2,
    },
    "temporal": {
        "seq_len": 30, "hidden_size": 64, "num_layers": 1, "dropout": 0.2, "bidirectional": False,
        "batch_size": 128, "epochs": 25, "lr": 1e-3, "weight_decay": 1e-4,
        "n_train": 12000, "n_val": 2000, "n_test": 3000, "early_stopping_patience": 5,
    },
    "decision": {
        "window_size": 30, "min_frames": 10, "drowsy_closed_ratio": 0.6, "drowsy_consecutive_closed": 12,
        "eye_closed_conf": 0.6, "distraction_ratio": 0.6, "distraction_conf": 0.5, "critical_ratio": 0.85,
        "temporal_model_weight": 0.5,
    },
    "gradcam_per_group": 6,
}
if QUICK_RUN:
    CFG["distraction"].update(epochs=2, batch_size=16, early_stopping_patience=2)
    CFG["eye_state"].update(epochs=2, batch_size=32)
    CFG["temporal"].update(epochs=3, n_train=600, n_val=150, n_test=300)
    CFG["gradcam_per_group"] = 2
    CFG["max_images_per_split"] = 400  # subsample for speed

def set_seed(seed: int) -> None:
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.benchmark = True       # fixed input sizes -> faster convolutions
    torch.backends.cudnn.deterministic = False  # full determinism costs speed; seeds still fix splits/init

set_seed(CFG["seed"])
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
NUM_WORKERS = min(4, os.cpu_count() or 1) if DEVICE.type == "cuda" else 0
N_GPUS = torch.cuda.device_count()

print("Device:", DEVICE, "| GPUs:", N_GPUS, "| AMP:", USE_AMP, "| DataLoader workers:", NUM_WORKERS)
for i in range(N_GPUS):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {p.name}, {p.total_memory / 1e9:.1f} GB")
if DEVICE.type == "cpu":
    print("WARNING: no GPU detected. Training will be very slow. Enable a GPU under Settings → Accelerator.")
print("QUICK_RUN:", QUICK_RUN, "| results ->", RESULTS)
(RESULTS / "config.json").write_text(json.dumps({"CFG": CFG, "QUICK_RUN": QUICK_RUN, "versions": versions,
                                                  "device": str(DEVICE), "n_gpus": N_GPUS}, indent=2))

## 3. Dataset discovery and validation

Datasets are located by **content**, not by path, because Kaggle mount points vary (`/kaggle/input/<slug>`, `/kaggle/input/competitions/<slug>`, `/kaggle/input/datasets/<owner>/<slug>`, …):

* **State Farm**: a `driver_imgs_list.csv` plus a `train` directory containing `c0 … c9`.
* **MRL Eye**: a directory containing `awake` and `sleepy` class folders, either inside `train/val/test` split folders (the usual Kaggle variant) or directly as a flat layout, which is then split here.

The folders are named `awake` / `sleepy`, but they contain **eye-state** labels (open / closed eyes). This notebook maps `awake → open` and `sleepy → closed`. A single closed eye is **not** a drowsiness label.

In [ ]:
DISTRACTION_CLASSES = [f"c{i}" for i in range(10)]
DISTRACTION_NAMES = {"c0": "Safe driving", "c1": "Texting - right", "c2": "Talking on phone - right",
                     "c3": "Texting - left", "c4": "Talking on phone - left", "c5": "Operating the radio",
                     "c6": "Drinking", "c7": "Reaching behind", "c8": "Hair and makeup", "c9": "Talking to passenger"}
MRL_FOLDER_TO_EYE_STATE = {"awake": "open", "sleepy": "closed"}
EYE_STATES = ["open", "closed"]
TEMPORAL_CLASSES = ["SAFE", "DISTRACTED", "DROWSY"]
IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp")


def walk_dirs(root: Path, max_depth: int = 7):
    # Yield (dirpath, subdirnames, filenames) without descending into image-heavy class folders.
    root = Path(root)
    base_depth = len(root.parts)
    for dirpath, dirnames, filenames in os.walk(root, followlinks=True):
        depth = len(Path(dirpath).parts) - base_depth
        yield Path(dirpath), dirnames, filenames
        names = set(dirnames)
        # stop below class-level folders: their children are images, not datasets
        if depth >= max_depth or names & set(DISTRACTION_CLASSES) or names & set(MRL_FOLDER_TO_EYE_STATE):
            dirnames[:] = [d for d in dirnames if d not in DISTRACTION_CLASSES and d not in MRL_FOLDER_TO_EYE_STATE]
        dirnames[:] = [d for d in dirnames if not d.startswith(".")]


def discover_datasets(input_root: Path) -> dict:
    if not input_root.exists():
        raise FileNotFoundError(f"Input root {input_root} does not exist. Attach the datasets via 'Add Input'.")
    sf_csv, sf_train, mrl_split_root, mrl_flat_root = [], [], [], []
    for d, dirnames, filenames in walk_dirs(input_root):
        names = set(dirnames)
        if "driver_imgs_list.csv" in filenames:
            sf_csv.append(d / "driver_imgs_list.csv")
        if d.name == "train" and set(DISTRACTION_CLASSES) <= names:
            sf_train.append(d)
        if {"train", "val", "test"} <= names and all(
                set(MRL_FOLDER_TO_EYE_STATE) <= {p.name for p in (d / s).iterdir() if p.is_dir()}
                for s in ("train", "val", "test")):
            mrl_split_root.append(d)
        elif set(MRL_FOLDER_TO_EYE_STATE) <= names and d.name not in ("train", "val", "test"):
            mrl_flat_root.append(d)
    found = {"state_farm_csv": sf_csv, "state_farm_train": sf_train,
             "mrl_split_root": mrl_split_root, "mrl_flat_root": mrl_flat_root}
    print(json.dumps({k: [str(p) for p in v] for k, v in found.items()}, indent=2))

    errors = []
    if not sf_csv or not sf_train:
        errors.append("State Farm not found: need driver_imgs_list.csv and a train/ folder with c0..c9. "
                      "Attach the 'State Farm Distracted Driver Detection' competition data (join the competition first).")
    if not mrl_split_root and not mrl_flat_root:
        errors.append("MRL Eye not found: need a folder with 'awake' and 'sleepy' sub-folders "
                      "(optionally inside train/val/test). Attach e.g. akashshingha850/mrl-eye-dataset.")
    if errors:
        raise FileNotFoundError("\n".join(errors))
    # Prefer the train dir that sits next to the CSV's imgs/ folder
    csv = sf_csv[0]
    train_dir = next((t for t in sf_train if csv.parent in t.parents), sf_train[0])
    return {"state_farm_csv": csv, "state_farm_train": train_dir,
            "mrl_root": (mrl_split_root or mrl_flat_root)[0], "mrl_has_splits": bool(mrl_split_root)}


# Optional: restore checkpoints from a previous run attached as an input (enables cross-session resume)
def restore_previous_checkpoints(input_root: Path) -> int:
    n = 0
    for d, dirnames, filenames in walk_dirs(input_root, max_depth=5):
        if d.name == "checkpoints" and d.parent.name == "DriveAwareAI_results":
            for f in filenames:
                if f.endswith("_last.pth") and not (DIRS["checkpoints"] / f).exists():
                    shutil.copy2(d / f, DIRS["checkpoints"] / f); n += 1
    return n


PATHS = discover_datasets(INPUT_ROOT)
print("Restored checkpoints from previous run:", restore_previous_checkpoints(INPUT_ROOT))
PATHS

In [ ]:
def load_state_farm(csv_path: Path, train_dir: Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    missing_cols = {"subject", "classname", "img"} - set(df.columns)
    if missing_cols:
        raise ValueError(f"{csv_path} is missing columns {missing_cols}; found {list(df.columns)}")
    unknown = set(df["classname"]) - set(DISTRACTION_CLASSES)
    if unknown:
        raise ValueError(f"Unexpected class names in CSV: {unknown}")
    df["path"] = [str(train_dir / c / f) for c, f in zip(df["classname"], df["img"])]
    exists = df["path"].map(os.path.isfile)
    n_missing = int((~exists).sum())
    df = df[exists].reset_index(drop=True)
    df["label"] = df["classname"].map({c: i for i, c in enumerate(DISTRACTION_CLASSES)})
    on_disk = sum(len([f for f in os.listdir(train_dir / c) if f.lower().endswith(IMG_EXT)]) for c in DISTRACTION_CLASSES)
    print(f"State Farm: {len(df)} labelled images found ({n_missing} CSV rows without a file, "
          f"{on_disk} images on disk), {df.subject.nunique()} drivers")
    if n_missing:
        print(f"  WARNING: {n_missing} rows in driver_imgs_list.csv have no matching image and were dropped.")
    if len(df) == 0:
        raise RuntimeError("No State Farm images could be matched to driver_imgs_list.csv")
    return df


def load_mrl(root: Path, has_splits: bool, seed: int) -> pd.DataFrame:
    rows = []
    splits = ("train", "val", "test") if has_splits else (None,)
    for s in splits:
        base = root / s if s else root
        for folder, state in MRL_FOLDER_TO_EYE_STATE.items():
            files = sorted(p for p in (base / folder).iterdir() if p.suffix.lower() in IMG_EXT)
            if not files:
                raise RuntimeError(f"MRL folder {base / folder} contains no images")
            rows += [{"path": str(p), "folder": folder, "eye_state": state, "label": EYE_STATES.index(state),
                      "split": s} for p in files]
        extra = {p.name for p in base.iterdir() if p.is_dir()} - set(MRL_FOLDER_TO_EYE_STATE)
        if extra:
            print(f"  NOTE: ignoring unexpected folders in {base}: {sorted(extra)}")
    df = pd.DataFrame(rows)
    # MRL file names usually start with a subject id (e.g. s0012_...). Use it when present.
    df["subject"] = df["path"].map(lambda p: (re.match(r"(s\d{3,4})_", Path(p).name) or [None, None])[1])
    if not has_splits:
        print("MRL has no predefined splits -> creating subject-grouped (or stratified random) 70/15/15 split")
        groups = df["subject"].fillna(df.index.to_series().astype(str))
        gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=seed)
        tv, te = next(gss.split(df, groups=groups))
        df.loc[df.index[te], "split"] = "test"
        tvdf = df.iloc[tv]
        gss2 = GroupShuffleSplit(n_splits=1, test_size=0.15 / 0.85, random_state=seed)
        tr, va = next(gss2.split(tvdf, groups=groups.iloc[tv]))
        df.loc[tvdf.index[tr], "split"] = "train"
        df.loc[tvdf.index[va], "split"] = "val"
    print("MRL eye images per split / state:")
    print(df.pivot_table(index="split", columns="eye_state", values="path", aggfunc="count").fillna(0).astype(int))
    if df["subject"].notna().mean() > 0.9:
        subj = {s: set(df[df.split == s].subject) for s in ("train", "val", "test")}
        overlap = len(subj["train"] & subj["test"])
        print(f"MRL subjects: train {len(subj['train'])}, val {len(subj['val'])}, test {len(subj['test'])}; "
              f"train∩test = {overlap}")
        if overlap:
            print("  WARNING: the predefined MRL split shares subjects between train and test -> eye-state test "
                  "accuracy is optimistic for unseen people.")
    else:
        print("MRL subject ids are not encoded in file names -> subject leakage cannot be checked.")
    return df


sf_df = load_state_farm(PATHS["state_farm_csv"], PATHS["state_farm_train"])
mrl_df = load_mrl(PATHS["mrl_root"], PATHS["mrl_has_splits"], CFG["seed"])

## 4. Dataset statistics and visualization

The cells below show class distributions, images per driver, sample images per class, and an image-integrity check (decode, size and colour mode on a random sample). All figures are saved to `plots/`.

In [ ]:
def image_quality(paths, n=300, seed=0) -> dict:
    rng = np.random.default_rng(seed)
    pick = rng.choice(len(paths), size=min(n, len(paths)), replace=False)
    sizes, modes, broken = Counter(), Counter(), []
    for i in pick:
        try:
            with Image.open(paths[i]) as im:
                im.verify()
            with Image.open(paths[i]) as im:
                sizes[f"{im.size[0]}x{im.size[1]}"] += 1; modes[im.mode] += 1
        except Exception as exc:
            broken.append(f"{paths[i]}: {exc}")
    return {"checked": int(len(pick)), "broken": broken, "top_sizes": sizes.most_common(5), "modes": dict(modes)}


sf_counts = sf_df.classname.value_counts().reindex(DISTRACTION_CLASSES).fillna(0).astype(int)
per_driver = sf_df.groupby("subject").size()
DATA_STATS = {
    "state_farm": {"images": len(sf_df), "drivers": int(sf_df.subject.nunique()),
                   "class_counts": sf_counts.to_dict(),
                   "imbalance_max_min": float(sf_counts.max() / max(1, sf_counts.min())),
                   "images_per_driver": per_driver.describe().round(1).to_dict(),
                   "quality": image_quality(sf_df.path.tolist())},
    "mrl": {"images": len(mrl_df),
            "counts": {s: mrl_df[mrl_df.split == s].eye_state.value_counts().to_dict() for s in ("train", "val", "test")},
            "quality": image_quality(mrl_df.path.tolist())},
}
(DIRS["reports"] / "dataset_stats.json").write_text(json.dumps(DATA_STATS, indent=2, default=str))
for name in ("state_farm", "mrl"):
    q = DATA_STATS[name]["quality"]
    print(f"{name}: checked {q['checked']} images, broken {len(q['broken'])}, sizes {q['top_sizes']}, modes {q['modes']}")
    if q["broken"]:
        print("  broken examples:", q["broken"][:5])

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
axes[0].bar([f"{c}\n{DISTRACTION_NAMES[c][:12]}" for c in DISTRACTION_CLASSES], sf_counts.values, color="tab:blue")
axes[0].set_title(f"State Farm classes (max/min = {DATA_STATS['state_farm']['imbalance_max_min']:.2f})")
axes[0].tick_params(axis="x", labelsize=7)
axes[1].bar(per_driver.index, per_driver.values, color="tab:green")
axes[1].set_title("State Farm images per driver"); axes[1].tick_params(axis="x", rotation=90, labelsize=7)
mrl_ct = mrl_df.groupby(["split", "eye_state"]).size().unstack().reindex(["train", "val", "test"])
mrl_ct.plot(kind="bar", ax=axes[2], color=["tab:orange", "tab:purple"]); axes[2].set_title("MRL eye states per split")
plt.tight_layout(); plt.savefig(DIRS["plots"] / "dataset_distributions.png", dpi=120); plt.show()

fig, axes = plt.subplots(2, 5, figsize=(16, 6.5))
for ax, c in zip(axes.ravel(), DISTRACTION_CLASSES):
    ax.imshow(Image.open(sf_df[sf_df.classname == c].path.iloc[0]).convert("RGB"))
    ax.set_title(f"{c}: {DISTRACTION_NAMES[c]}", fontsize=9); ax.axis("off")
plt.tight_layout(); plt.savefig(DIRS["plots"] / "state_farm_samples.png", dpi=110); plt.show()

fig, axes = plt.subplots(2, 6, figsize=(12, 4.5))
for row, state in zip(axes, EYE_STATES):
    for ax, p in zip(row, mrl_df[(mrl_df.eye_state == state) & (mrl_df.split == "train")].path.head(6)):
        ax.imshow(Image.open(p).convert("RGB")); ax.set_title(state, fontsize=9); ax.axis("off")
plt.tight_layout(); plt.savefig(DIRS["plots"] / "mrl_samples.png", dpi=110); plt.show()

## 5. Preprocessing and augmentation

* **Evaluation and inference:** resize, convert to RGB, then ImageNet mean/std normalisation. The same transform is used for validation, test and Grad-CAM.
* **State Farm training:** RandomResizedCrop(0.8–1.0), rotation ±8°, colour jitter, 5% grayscale, random erasing.
  **No horizontal flip.** Mirroring turns *texting – right* (c1) into an image of *texting – left* (c3) while keeping the c1 label. The original notebook flipped images; this pipeline does not.
* **MRL training:** horizontal flip (label-preserving for eyes), rotation ±10°, brightness/contrast jitter.
* **Class imbalance:** inverse-frequency class weights in the cross-entropy loss, normalised to mean 1.

In [ ]:
IMAGENET_MEAN, IMAGENET_STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

def eval_transform(size):
    return transforms.Compose([transforms.Resize((size, size)), transforms.ToTensor(),
                               transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)])

def distraction_train_transform(size):
    return transforms.Compose([
        transforms.RandomResizedCrop(size, scale=(0.8, 1.0), ratio=(0.9, 1.1)),
        transforms.RandomRotation(8),
        transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.15, hue=0.02),
        transforms.RandomGrayscale(p=0.05),
        transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        transforms.RandomErasing(p=0.25, scale=(0.02, 0.12))])

def eye_train_transform(size):
    return transforms.Compose([
        transforms.Resize((size, size)), transforms.RandomHorizontalFlip(), transforms.RandomRotation(10),
        transforms.ColorJitter(brightness=0.3, contrast=0.3),
        transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)])

assert "RandomHorizontalFlip" not in [type(t).__name__ for t in distraction_train_transform(224).transforms]

def denormalize(t: torch.Tensor) -> np.ndarray:
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1); std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return (t.detach().cpu() * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()

def class_weights(labels, n):
    counts = np.bincount(np.asarray(labels), minlength=n).astype(float)
    w = counts.sum() / (n * np.clip(counts, 1, None))
    return torch.tensor(w / w.mean(), dtype=torch.float32)

class ImageDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform):
        self.paths = df["path"].tolist(); self.labels = df["label"].astype(int).tolist(); self.transform = transform
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        return self.transform(Image.open(self.paths[i]).convert("RGB")), self.labels[i]

# Preview augmentations
x0 = Image.open(sf_df.path.iloc[0]).convert("RGB")
fig, axes = plt.subplots(1, 5, figsize=(16, 3.2))
axes[0].imshow(x0.resize((224, 224))); axes[0].set_title("original")
for ax in axes[1:]:
    ax.imshow(denormalize(distraction_train_transform(224)(x0))); ax.set_title("augmented")
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.savefig(DIRS["plots"] / "augmentation_preview.png", dpi=100); plt.show()

## 6. Driver-aware train / validation / test split (State Farm)

State Farm images are frames from videos, so images of the same driver are near-duplicates. A random split would let the model "recognise" test drivers and inflate accuracy. Instead, we split **by driver** (`subject`) with `GroupShuffleSplit`: about 15% of drivers for test, about 15% for validation, and the rest for training. An assertion guarantees no driver is shared between splits. The test split is used **only** for the final evaluation, never for model selection.

For MRL, the dataset's predefined `train/val/test` folders are used when present (the overlap check in Section 3 reports whether subjects leak between them).

In [ ]:
def group_split(df, group_col, val_fraction, test_fraction, seed):
    gss = GroupShuffleSplit(n_splits=1, test_size=test_fraction, random_state=seed)
    tv_idx, te_idx = next(gss.split(df, groups=df[group_col]))
    tv, te = df.iloc[tv_idx], df.iloc[te_idx]
    gss2 = GroupShuffleSplit(n_splits=1, test_size=val_fraction / (1 - test_fraction), random_state=seed)
    tr_idx, va_idx = next(gss2.split(tv, groups=tv[group_col]))
    tr, va = tv.iloc[tr_idx], tv.iloc[va_idx]
    a, b, c = set(tr[group_col]), set(va[group_col]), set(te[group_col])
    assert not (a & b or a & c or b & c), "driver leakage between splits"
    return tr.reset_index(drop=True), va.reset_index(drop=True), te.reset_index(drop=True)

dc = CFG["distraction"]
sf_train, sf_val, sf_test = group_split(sf_df, "subject", dc["val_fraction"], dc["test_fraction"], CFG["seed"])
mrl_train = mrl_df[mrl_df.split == "train"].reset_index(drop=True)
mrl_val = mrl_df[mrl_df.split == "val"].reset_index(drop=True)
mrl_test = mrl_df[mrl_df.split == "test"].reset_index(drop=True)

if CFG.get("max_images_per_split"):
    cap = CFG["max_images_per_split"]
    sample = lambda d: d.sample(min(len(d), cap), random_state=CFG["seed"]).reset_index(drop=True)
    sf_train, sf_val, sf_test = map(sample, (sf_train, sf_val, sf_test))
    mrl_train, mrl_val, mrl_test = map(sample, (mrl_train, mrl_val, mrl_test))
    print(f"QUICK_RUN: each split capped at {cap} images")

SPLITS = {"state_farm": {s: {"images": len(d), "drivers": sorted(d.subject.unique().tolist())}
                         for s, d in (("train", sf_train), ("val", sf_val), ("test", sf_test))},
          "mrl": {s: len(d) for s, d in (("train", mrl_train), ("val", mrl_val), ("test", mrl_test))}}
(DIRS["reports"] / "splits.json").write_text(json.dumps(SPLITS, indent=2))
for s, v in SPLITS["state_farm"].items():
    print(f"State Farm {s:5s}: {v['images']:6d} images, {len(v['drivers'])} drivers {v['drivers']}")
print("MRL:", SPLITS["mrl"])
for d, name in ((sf_train, "train"), (sf_val, "val"), (sf_test, "test")):
    missing = set(range(10)) - set(d.label)
    if missing:
        print(f"  WARNING: State Farm {name} split has no images of classes {sorted(missing)}")

## 7. Models, training loop and evaluation utilities

These are the project's components:

* `build_resnet_classifier`: ResNet-18 with `fc = Sequential(Dropout, Linear)`. Its state dicts are compatible with the project's checkpoints.
* `apply_freeze_mode`: `frozen` (head only), `finetune_layer4`, `finetune_layer3_4`, or `full`.
* `fit`: AdamW, AMP on CUDA, LR scheduling, early stopping on validation loss, and checkpointing:
  * `<run>_last.pth` after **every** epoch (model, optimizer, scheduler, scaler, history, early-stopping state). Training resumes from it automatically.
  * `<run>_best.pth` whenever validation loss improves. The best weights are restored at the end.
* `evaluate_classifier`: accuracy, macro/weighted precision, recall and F1, per-class metrics, ROC-AUC, confusion matrices, classification report, and a misclassified-image grid.

In [ ]:
FREEZE_MODES = ("frozen", "finetune_layer4", "finetune_layer3_4", "full")

def build_resnet_classifier(num_classes, dropout, pretrained=True):
    weights = models.ResNet18_Weights.DEFAULT if pretrained else None
    try:
        m = models.resnet18(weights=weights)
    except Exception as exc:
        raise RuntimeError("Could not download ImageNet ResNet-18 weights. Turn Internet ON in the notebook "
                           f"settings (Settings → Internet). Original error: {exc}") from exc
    m.fc = nn.Sequential(nn.Dropout(dropout), nn.Linear(m.fc.in_features, num_classes))
    return m

def apply_freeze_mode(model, mode):
    if mode not in FREEZE_MODES:
        raise ValueError(f"freeze mode must be one of {FREEZE_MODES}")
    trainable = {"frozen": ["fc"], "finetune_layer4": ["layer4", "fc"], "finetune_layer3_4": ["layer3", "layer4", "fc"]}.get(mode)
    for name, p in model.named_parameters():
        p.requires_grad = trainable is None or any(name.startswith(t) for t in trainable)
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def run_epoch(model, loader, criterion, optimizer=None, scaler=None):
    train = optimizer is not None
    model.train(train)
    total, correct, n = 0.0, 0, 0
    with torch.set_grad_enabled(train):
        for x, y in loader:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
                logits = model(x)
                loss = criterion(logits, y)
            if not torch.isfinite(loss):
                raise FloatingPointError(f"Non-finite loss {loss.item()}; lower the learning rate.")
            if train:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            total += loss.item() * y.size(0); correct += (logits.argmax(1) == y).sum().item(); n += y.size(0)
    return total / max(n, 1), correct / max(n, 1)


def fit(run_name, model, train_loader, val_loader, criterion, optimizer, scheduler, epochs, patience,
        meta, val_criterion=None):
    # Train with per-epoch checkpoints, resume, early stopping on val loss; returns history.
    val_criterion = val_criterion or criterion
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    last_p, best_p = DIRS["checkpoints"] / f"{run_name}_last.pth", DIRS["checkpoints"] / f"{run_name}_best.pth"
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": [], "lr": [], "epoch_seconds": []}
    start_epoch, best_val, bad_epochs, best_epoch, done = 1, float("inf"), 0, 0, False
    if last_p.exists():
        ck = torch.load(last_p, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ck["model_state_dict"]); optimizer.load_state_dict(ck["optimizer"])
        if scheduler is not None and ck.get("scheduler"): scheduler.load_state_dict(ck["scheduler"])
        if ck.get("scaler"): scaler.load_state_dict(ck["scaler"])
        history, best_val, bad_epochs, best_epoch = ck["history"], ck["best_val"], ck["bad_epochs"], ck["best_epoch"]
        start_epoch, done = ck["epoch"] + 1, ck.get("finished", False)
        print(f"[{run_name}] resuming after epoch {ck['epoch']} (best val loss {best_val:.4f} @ epoch {best_epoch})"
              + (" - already finished" if done else ""))
    t_start = time.time()
    for epoch in range(start_epoch, epochs + 1):
        if done: break
        t0 = time.time()
        tr_loss, tr_acc = run_epoch(model, train_loader, criterion, optimizer, scaler)
        va_loss, va_acc = run_epoch(model, val_loader, val_criterion)
        if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau): scheduler.step(va_loss)
        elif scheduler is not None: scheduler.step()
        for k, v in zip(history, (tr_loss, va_loss, tr_acc, va_acc, optimizer.param_groups[0]["lr"], time.time() - t0)):
            history[k].append(v)
        improved = va_loss < best_val - 1e-4
        if improved:
            best_val, best_epoch, bad_epochs = va_loss, epoch, 0
            torch.save({"model_state_dict": model.state_dict(), "epoch": epoch, "val_loss": va_loss,
                        "val_acc": va_acc, **meta}, best_p)
        else:
            bad_epochs += 1
        done = bad_epochs >= patience or epoch == epochs
        torch.save({"model_state_dict": model.state_dict(), "optimizer": optimizer.state_dict(),
                    "scheduler": scheduler.state_dict() if scheduler is not None else None,
                    "scaler": scaler.state_dict(), "history": history, "epoch": epoch, "best_val": best_val,
                    "bad_epochs": bad_epochs, "best_epoch": best_epoch, "finished": done}, last_p)
        print(f"[{run_name}] epoch {epoch:02d}/{epochs} | train loss {tr_loss:.4f} acc {tr_acc:.4f} | "
              f"val loss {va_loss:.4f} acc {va_acc:.4f} | {history['epoch_seconds'][-1]:.0f}s" + ("  *best*" if improved else ""))
        if bad_epochs >= patience:
            print(f"[{run_name}] early stopping (no val-loss improvement for {patience} epochs)")
    if not best_p.exists():
        raise RuntimeError(f"[{run_name}] no best checkpoint was written; training did not complete an epoch")
    best = torch.load(best_p, map_location=DEVICE, weights_only=False)
    model.load_state_dict(best["model_state_dict"])
    (DIRS["history"] / f"{run_name}.json").write_text(json.dumps(history, indent=2))
    plot_history(history, run_name)
    return {"history": history, "best_epoch": best["epoch"], "best_val_loss": best["val_loss"],
            "best_val_acc": best["val_acc"], "train_seconds_this_session": time.time() - t_start,
            "train_seconds_total": float(sum(history["epoch_seconds"]))}


def plot_history(h, name):
    ep = np.arange(1, len(h["train_loss"]) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
    for ax, k in zip(axes, ("loss", "acc")):
        ax.plot(ep, h[f"train_{k}"], "o-", label="train"); ax.plot(ep, h[f"val_{k}"], "o-", label="validation")
        ax.set_xlabel("epoch"); ax.set_ylabel(k); ax.grid(alpha=.3); ax.legend()
    fig.suptitle(f"{name}: training curves"); plt.tight_layout()
    plt.savefig(DIRS["plots"] / f"{name}_curves.png", dpi=120); plt.show()

In [ ]:
def compute_metrics(y, pred, prob, labels):
    idx = list(range(len(labels)))
    p, r, f, s = precision_recall_fscore_support(y, pred, labels=idx, zero_division=0)
    out = {"accuracy": float(accuracy_score(y, pred)), "n_samples": int(len(y))}
    for avg in ("macro", "weighted"):
        ap, ar, af, _ = precision_recall_fscore_support(y, pred, labels=idx, average=avg, zero_division=0)
        out.update({f"{avg}_precision": float(ap), f"{avg}_recall": float(ar), f"{avg}_f1": float(af)})
    out["per_class"] = {labels[i]: {"precision": float(p[i]), "recall": float(r[i]), "f1": float(f[i]),
                                    "support": int(s[i])} for i in idx}
    if prob is not None and len(np.unique(y)) == len(labels):
        out["roc_auc"] = float(roc_auc_score(y, prob[:, 1]) if len(labels) == 2
                               else roc_auc_score(y, prob, multi_class="ovr", average="macro"))
    elif prob is not None:
        out["roc_auc"] = None
        out["roc_auc_note"] = "not computed: some classes absent from the test labels"
    return out


def plot_confusion(cm, labels, name, title):
    fig, axes = plt.subplots(1, 2, figsize=(2 * (1.5 + 0.75 * len(labels)), 1.2 + 0.7 * len(labels)))
    norm = cm / np.clip(cm.sum(1, keepdims=True), 1, None)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=labels, yticklabels=labels, ax=axes[0])
    sns.heatmap(norm, annot=True, fmt=".2f", cmap="Blues", xticklabels=labels, yticklabels=labels, ax=axes[1], vmin=0, vmax=1)
    for ax, t in zip(axes, ("counts", "row-normalised")):
        ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title(f"{title} ({t})")
    plt.tight_layout(); plt.savefig(DIRS["plots"] / f"{name}_confusion_matrix.png", dpi=120); plt.show()


@torch.no_grad()
def predict(model, loader):
    model.eval(); ys, probs = [], []
    for x, y in loader:
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            logits = model(x.to(DEVICE, non_blocking=True))
        probs.append(torch.softmax(logits.float(), 1).cpu().numpy()); ys.append(y.numpy())
    prob = np.concatenate(probs)
    return np.concatenate(ys), prob.argmax(1), prob


def misclassified_grid(dataset, y, pred, prob, labels, name, n=16):
    wrong = np.flatnonzero(y != pred)
    if len(wrong) == 0:
        print("No misclassified images."); return
    pick = np.random.default_rng(0).choice(wrong, size=min(n, len(wrong)), replace=False)
    cols = 4; rows = math.ceil(len(pick) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(3.2 * cols, 3.3 * rows))
    for ax in np.atleast_1d(axes).ravel(): ax.axis("off")
    for ax, i in zip(np.atleast_1d(axes).ravel(), pick):
        ax.imshow(denormalize(dataset[int(i)][0]))
        ax.set_title(f"true {labels[y[i]]} / pred {labels[pred[i]]}\nconf {prob[i, pred[i]]:.2f}", fontsize=8)
    plt.tight_layout(); plt.savefig(DIRS["plots"] / f"{name}_misclassified.png", dpi=110); plt.show()


def measure_latency(model, shape, iters=50, warmup=10):
    model.eval(); x = torch.randn(*shape, device=DEVICE); times = []
    with torch.no_grad():
        for i in range(warmup + iters):
            if DEVICE.type == "cuda": torch.cuda.synchronize()
            t0 = time.perf_counter(); model(x)
            if DEVICE.type == "cuda": torch.cuda.synchronize()
            if i >= warmup: times.append((time.perf_counter() - t0) * 1000)
    return {"device": str(DEVICE), "batch": shape[0], "median_ms": float(np.median(times)), "p95_ms": float(np.percentile(times, 95))}


def evaluate_classifier(model, df, size, labels, name, title):
    ds = ImageDataset(df, eval_transform(size))
    loader = DataLoader(ds, batch_size=128, shuffle=False, num_workers=NUM_WORKERS, pin_memory=USE_AMP)
    t0 = time.time(); y, pred, prob = predict(model, loader); secs = time.time() - t0
    metrics = compute_metrics(y, pred, prob, labels)
    metrics["eval_seconds"] = secs
    metrics["latency_batch1"] = measure_latency(model, (1, 3, size, size))
    report = classification_report(y, pred, labels=list(range(len(labels))), target_names=labels, digits=4, zero_division=0)
    cm = confusion_matrix(y, pred, labels=list(range(len(labels))))
    print(f"=== {title} ===\n{report}")
    print(f"ROC-AUC: {metrics.get('roc_auc')} | batch-1 latency {metrics['latency_batch1']['median_ms']:.2f} ms")
    plot_confusion(cm, labels, name, title)
    misclassified_grid(ds, y, pred, prob, labels, name)
    (DIRS["reports"] / f"{name}_metrics.json").write_text(json.dumps({**metrics, "confusion_matrix": cm.tolist()}, indent=2))
    (DIRS["reports"] / f"{name}_classification_report.txt").write_text(report)
    return metrics, ds, (y, pred, prob)

def loader_for(df, transform, batch_size, shuffle):
    return DataLoader(ImageDataset(df, transform), batch_size=batch_size, shuffle=shuffle, drop_last=shuffle,
                      num_workers=NUM_WORKERS, pin_memory=USE_AMP, persistent_workers=NUM_WORKERS > 0)

## 8. Distraction detection: ResNet-18, frozen vs. fine-tuned

Two configurations are trained with identical data, seed and schedule, differing only in which layers are trainable:

| Run | Trainable layers |
|---|---|
| `frozen` | classification head only (ImageNet features as-is) |
| `finetune_layer3_4` | layer3 + layer4 + head (project default) |

Loss: class-weighted cross-entropy with label smoothing 0.1 (validation uses plain CE, so losses are comparable). Optimiser: AdamW with head LR 3e-4, backbone LR 1e-4, and cosine annealing. Early stopping on validation loss. Each model is evaluated on the **held-out drivers** test split.

In [ ]:
RESULTS_SUMMARY = {"distraction": {}, "eye_state": {}, "temporal": {}}
TRAINED = {}

dist_train_loader = loader_for(sf_train, distraction_train_transform(dc["image_size"]), dc["batch_size"], True)
dist_val_loader = loader_for(sf_val, eval_transform(dc["image_size"]), dc["batch_size"], False)
dist_weights = class_weights(sf_train.label, 10).to(DEVICE)
print("class weights:", dist_weights.cpu().numpy().round(3))

for mode in dc["freeze_modes"]:
    run = f"distraction_{mode}"
    set_seed(CFG["seed"])
    model = build_resnet_classifier(10, dc["dropout"]).to(DEVICE)
    n_train = apply_freeze_mode(model, mode)
    head = [p for n, p in model.named_parameters() if n.startswith("fc") and p.requires_grad]
    body = [p for n, p in model.named_parameters() if not n.startswith("fc") and p.requires_grad]
    groups = [{"params": head, "lr": dc["lr_head"]}] + ([{"params": body, "lr": dc["lr_backbone"]}] if body else [])
    optimizer = torch.optim.AdamW(groups, weight_decay=dc["weight_decay"])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=dc["epochs"])
    criterion = nn.CrossEntropyLoss(weight=dist_weights, label_smoothing=dc["label_smoothing"])
    print(f"\n##### {run}: {n_train:,} trainable parameters #####")
    fit_res = fit(run, model, dist_train_loader, dist_val_loader, criterion, optimizer, scheduler, dc["epochs"],
                  dc["early_stopping_patience"], {"class_names": DISTRACTION_CLASSES, "image_size": dc["image_size"],
                  "freeze_mode": mode, "normalization_mean": IMAGENET_MEAN, "normalization_std": IMAGENET_STD},
                  val_criterion=nn.CrossEntropyLoss())
    metrics, test_ds, preds = evaluate_classifier(model, sf_test, dc["image_size"], DISTRACTION_CLASSES,
                                                  f"{run}_test", f"Distraction {mode} (held-out drivers)")
    RESULTS_SUMMARY["distraction"][mode] = {"trainable_params": n_train, **{k: v for k, v in fit_res.items() if k != "history"},
                                            **{k: v for k, v in metrics.items() if k != "per_class"}}
    TRAINED[run] = {"model": model, "val_loss": fit_res["best_val_loss"], "val_acc": fit_res["best_val_acc"],
                    "test_ds": test_ds, "preds": preds, "metrics": metrics}

## 9. Eye-state classification: ResNet-18 (MRL)

ResNet-18 with layer4 and the head fine-tuned, at 160 px, using class-weighted CE, AdamW (lr 1e-4) and ReduceLROnPlateau. The output is **open / closed eye state** with confidence. It is not drowsiness.

In [ ]:
ec = CFG["eye_state"]
run = f"eye_state_{ec['freeze_mode']}"
set_seed(CFG["seed"])
eye_model = build_resnet_classifier(2, ec["dropout"]).to(DEVICE)
n_train = apply_freeze_mode(eye_model, ec["freeze_mode"])
eye_opt = torch.optim.AdamW([p for p in eye_model.parameters() if p.requires_grad], lr=ec["lr"], weight_decay=ec["weight_decay"])
eye_sched = torch.optim.lr_scheduler.ReduceLROnPlateau(eye_opt, factor=0.5, patience=1)
eye_crit = nn.CrossEntropyLoss(weight=class_weights(mrl_train.label, 2).to(DEVICE))
print(f"##### {run}: {n_train:,} trainable parameters #####")
eye_fit = fit(run, eye_model, loader_for(mrl_train, eye_train_transform(ec["image_size"]), ec["batch_size"], True),
              loader_for(mrl_val, eval_transform(ec["image_size"]), ec["batch_size"], False), eye_crit, eye_opt, eye_sched,
              ec["epochs"], ec["early_stopping_patience"],
              {"class_names": EYE_STATES, "mrl_folder_mapping": MRL_FOLDER_TO_EYE_STATE, "image_size": ec["image_size"],
               "freeze_mode": ec["freeze_mode"], "normalization_mean": IMAGENET_MEAN, "normalization_std": IMAGENET_STD},
              val_criterion=nn.CrossEntropyLoss())
eye_metrics, eye_test_ds, eye_preds = evaluate_classifier(eye_model, mrl_test, ec["image_size"], EYE_STATES,
                                                          f"{run}_test", "Eye state (MRL test)")
RESULTS_SUMMARY["eye_state"][ec["freeze_mode"]] = {"trainable_params": n_train,
    **{k: v for k, v in eye_fit.items() if k != "history"}, **{k: v for k, v in eye_metrics.items() if k != "per_class"}}
TRAINED[run] = {"model": eye_model, "test_ds": eye_test_ds, "preds": eye_preds, "metrics": eye_metrics}

## 10. Model comparison and best-model selection

The best distraction configuration is chosen by **validation loss**, never by test metrics. Test metrics are shown for reporting only. The selected models are exported to `models/` in the same checkpoint format the DriveAwareAI app loads (`model_state_dict`, `class_names`, `image_size`, normalisation, metrics).

In [ ]:
rows = []
for task in ("distraction", "eye_state"):
    for cfg_name, r in RESULTS_SUMMARY[task].items():
        rows.append({"task": task, "config": cfg_name, "trainable_params": r["trainable_params"],
                     "best_epoch": r["best_epoch"], "val_loss": r["best_val_loss"], "val_acc": r["best_val_acc"],
                     "test_accuracy": r["accuracy"], "test_macro_precision": r["macro_precision"],
                     "test_macro_recall": r["macro_recall"], "test_macro_f1": r["macro_f1"],
                     "test_roc_auc": r.get("roc_auc"), "train_min": r["train_seconds_total"] / 60,
                     "latency_ms_b1": r["latency_batch1"]["median_ms"]})
comparison = pd.DataFrame(rows)
comparison.to_csv(DIRS["reports"] / "model_comparison.csv", index=False)
display(comparison.round(4)) if "display" in globals() else print(comparison.round(4).to_string())

best_mode = min(dc["freeze_modes"], key=lambda m: TRAINED[f"distraction_{m}"]["val_loss"])
print(f"Selected distraction model (lowest validation loss): {best_mode}")

fig, ax = plt.subplots(figsize=(8, 3.5))
sub = comparison[comparison.task == "distraction"].set_index("config")[["val_acc", "test_accuracy", "test_macro_f1"]]
sub.plot(kind="bar", ax=ax); ax.set_ylim(0, 1); ax.set_title("Distraction: frozen vs fine-tuned"); ax.grid(axis="y", alpha=.3)
plt.xticks(rotation=0); plt.tight_layout(); plt.savefig(DIRS["plots"] / "distraction_model_comparison.png", dpi=120); plt.show()

per_class = pd.DataFrame(TRAINED[f"distraction_{best_mode}"]["metrics"]["per_class"]).T
per_class.index = [f"{c} {DISTRACTION_NAMES[c]}" for c in per_class.index]
print("Per-class metrics of the selected distraction model (test):")
print(per_class.round(4).to_string())

def export(model, path, meta):
    torch.save({"model_state_dict": model.state_dict(), **meta}, path); return path

dm = TRAINED[f"distraction_{best_mode}"]["metrics"]
export(TRAINED[f"distraction_{best_mode}"]["model"], DIRS["models"] / "distraction_resnet18.pth",
       {"class_names": DISTRACTION_CLASSES, "image_size": dc["image_size"], "normalization_mean": IMAGENET_MEAN,
        "normalization_std": IMAGENET_STD, "freeze_mode": best_mode, "test_accuracy": dm["accuracy"],
        "macro_precision": dm["macro_precision"], "macro_recall": dm["macro_recall"], "macro_f1": dm["macro_f1"]})
export(eye_model, DIRS["models"] / "eye_state_resnet18.pth",
       {"class_names": ["awake", "sleepy"], "class_to_idx": {"awake": 0, "sleepy": 1}, "eye_states": EYE_STATES,
        "image_size": ec["image_size"], "normalization_mean": IMAGENET_MEAN, "normalization_std": IMAGENET_STD,
        "test_accuracy": eye_metrics["accuracy"], "macro_precision": eye_metrics["macro_precision"],
        "macro_recall": eye_metrics["macro_recall"], "macro_f1": eye_metrics["macro_f1"]})
print("Exported:", sorted(p.name for p in DIRS["models"].iterdir()))

## 11. Grad-CAM visualizations

Grad-CAM is computed on `layer4[-1]` (the last residual block, 7×7 at 224 px and 5×5 at 160 px), the deepest layer that still has spatial resolution. It's implemented here with forward and backward hooks, which matches the project's `pytorch-grad-cam` setup (ReLU on the gradient-weighted activations, min-max normalised) without an extra install.

For each model the notebook saves grids of **correct** and **incorrect** test predictions: the original image, then the heatmap overlay.

**Limitations.** Grad-CAM shows where the gradient of the predicted logit is large. It is not a causal explanation. Maps are coarse (upsampled from 7×7), can highlight spurious context (windows, seats, background), and a plausible map does not mean the prediction is right. For State Farm, check whether the maps focus on the hands, phone and head rather than the background.

In [ ]:
class GradCAM:
    def __init__(self, model):
        self.model = model.eval(); self.acts = self.grads = None
        layer = model.layer4[-1]
        self.h1 = layer.register_forward_hook(self._save)

    def _save(self, module, inp, out):
        # tensor hook (not a module backward hook): safe with the block's in-place ReLU
        self.acts = out
        out.register_hook(lambda g: setattr(self, "grads", g))

    def __call__(self, x, class_idx=None):
        x = x.to(DEVICE).requires_grad_(True)
        self.model.zero_grad(set_to_none=True)
        logits = self.model(x)
        prob = torch.softmax(logits, 1)[0].detach().cpu().numpy()
        cls = int(prob.argmax()) if class_idx is None else int(class_idx)
        logits[0, cls].backward()
        w = self.grads.mean(dim=(2, 3), keepdim=True)
        cam = F.relu((w * self.acts).sum(1)).squeeze(0)
        cam = F.interpolate(cam[None, None], size=x.shape[-2:], mode="bilinear", align_corners=False)[0, 0]
        cam = cam - cam.min(); cam = cam / (cam.max() + 1e-8)
        return cam.detach().cpu().numpy(), cls, prob

    def close(self):
        self.h1.remove()


def overlay(img, cam, alpha=0.45):
    heat = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)[..., ::-1] / 255.0
    return np.clip((1 - alpha) * img + alpha * heat, 0, 1)


def gradcam_grids(model, dataset, y, pred, labels, name, n_each):
    for p in model.parameters(): p.requires_grad_(True)   # frozen backbones still need grads w.r.t. activations
    cam = GradCAM(model)
    out = []
    try:
        for group, idx in (("correct", np.flatnonzero(y == pred)), ("incorrect", np.flatnonzero(y != pred))):
            if len(idx) == 0:
                print(f"{name}: no {group} predictions to visualise"); continue
            pick = np.random.default_rng(CFG["seed"]).choice(idx, size=min(n_each, len(idx)), replace=False)
            fig, axes = plt.subplots(len(pick), 2, figsize=(6.4, 3.1 * len(pick)))
            axes = np.atleast_2d(axes)
            for row, i in zip(axes, pick):
                x, label = dataset[int(i)]
                heat, cls, prob = cam(x.unsqueeze(0))
                img = denormalize(x)
                row[0].imshow(img); row[0].set_title(f"original (true: {labels[label]})", fontsize=8)
                row[1].imshow(overlay(img, heat)); row[1].set_title(f"Grad-CAM: pred {labels[cls]} ({prob[cls]:.2f})", fontsize=8)
                for a in row: a.axis("off")
            fig.suptitle(f"{name}: {group} predictions"); plt.tight_layout()
            path = DIRS["gradcam"] / f"{name}_{group}.png"
            plt.savefig(path, dpi=110); plt.show(); out.append(path)
    finally:
        cam.close()
    return out

best_d = TRAINED[f"distraction_{best_mode}"]
y, pred, _ = best_d["preds"]
gc_files = gradcam_grids(best_d["model"], best_d["test_ds"], y, pred, DISTRACTION_CLASSES,
                         f"distraction_{best_mode}", CFG["gradcam_per_group"])
y, pred, _ = eye_preds
gc_files += gradcam_grids(eye_model, eye_test_ds, y, pred, EYE_STATES, "eye_state", CFG["gradcam_per_group"])
print("Saved:", [p.name for p in gc_files])

## 12. Temporal model: LSTM + attention

### What this model is, and what it is not

The project's temporal head works on a rolling window of **T = 30 frames** (about 2 s at 15 FPS). Each frame is represented by the fused vector `[p(eye closed), p(c0), …, p(c9)]` produced by the two encoders above. The architecture is unchanged from the project:

`LayerNorm(11) → LSTM(hidden 64) → additive attention (score_t = vᵀ tanh(W h_t), α = softmax_t) → Dropout → Linear(64, 3)` for SAFE / DISTRACTED / DROWSY.

**Training data: simulated, not real.** State Farm images are shuffled stills and MRL images are isolated eye crops. Neither dataset contains ordered video frames or drowsiness labels, so a genuine supervised drowsiness sequence model **cannot** be trained or validated from them. As in the project, the temporal head is trained on **simulated sequences of encoder outputs**:

* a latent per-window state (the label) drives blink, long-closure/microsleep and sustained-distraction patterns;
* per-frame encoder errors are injected at the **error rates measured on the test sets in Sections 8–9**, plus random confidences, missing eye detections (0/10/50% of frames) and burst errors.

Its test metrics therefore only show that the model can **separate persistent patterns from frame-level noise under the simulator's assumptions**. They are **not** real-world drowsiness-detection performance and must not be reported as such.

### What genuine temporal validation would need
* **Ordered driver video** at a known frame rate, from an in-cabin camera similar to the deployment camera (RGB and/or NIR, day and night).
* **Time-aligned drowsiness labels**, for example per-segment alert / low-vigilance / drowsy from observer ratings or self-reports (KSS, Karolinska Sleepiness Scale), ideally backed by physiological or performance measures (PERCLOS from an eye tracker, EEG, lane-keeping or reaction time).
* **Per-frame or per-segment distraction annotations** aligned with the same video.
* **Subject-disjoint splits** and enough subjects to cover age, eyewear, skin tone and lighting.
* Public options include **NTHU-DDD** (driver drowsiness detection video), **UTA-RLDD** (Real-Life Drowsiness Dataset, alert/low-vigilant/drowsy videos of 60 subjects), **DMD** (Driver Monitoring Dataset, distraction and drowsiness video annotations), and **YawDD** (yawning). With such data, the encoders above would produce the per-frame vectors and this LSTM would be trained on real sequences.

In [ ]:
N_DIST = 10
eye_err = 1 - eye_metrics["accuracy"]
dist_err = 1 - TRAINED[f"distraction_{best_mode}"]["metrics"]["accuracy"]
print(f"Simulator noise uses measured test error rates: eye {eye_err:.4f}, distraction {dist_err:.4f}")


def _softmax_like(rng, true_idx, n, conf):
    rest = rng.dirichlet(np.ones(n - 1)) * (1 - conf)
    return np.insert(rest, true_idx, conf).astype(np.float32)

def _eye_closed_track(rng, T, drowsy):
    closed = np.zeros(T, dtype=bool); rate = rng.uniform(0.01, 0.05); t = 0
    while t < T:
        if rng.random() < rate:
            d = rng.integers(2, 7); closed[t:t + d] = True; t += d
        t += 1
    if drowsy:
        for _ in range(rng.integers(1, 4)):
            d = rng.integers(max(8, T // 4), max(9, int(T * 0.7))); s = rng.integers(0, max(1, T - d)); closed[s:s + d] = True
        if closed.mean() < 0.4:
            s = rng.integers(0, T // 3); closed[s:s + int(T * 0.5)] = True
    return closed

def _distraction_track(rng, T, distracted):
    track = np.zeros(T, dtype=int)
    if distracted:
        cls = rng.integers(1, N_DIST); s = rng.integers(0, T // 4); e = rng.integers(min(T, s + int(T * 0.6)), T + 1)
        track[s:e] = cls
    for _ in range(rng.integers(0, 3)):
        d = rng.integers(1, 6); s = rng.integers(0, T); track[s:s + d] = rng.integers(1, N_DIST)
    return track

def simulate_sequence(rng, T, label, eye_err, dist_err):
    closed = _eye_closed_track(rng, T, label == 2); behaviour = _distraction_track(rng, T, label == 1)
    eye_missing_p = rng.choice([0.0, 0.1, 0.5]); burst = rng.random() < 0.2
    seq = np.zeros((T, 1 + N_DIST), dtype=np.float32)
    for t in range(T):
        if rng.random() < eye_missing_p:
            seq[t, 0] = 0.5          # no eye detected -> uninformative
        else:
            is_closed = closed[t] ^ (rng.random() < eye_err * (5 if burst else 1))
            conf = rng.uniform(0.6, 1.0); seq[t, 0] = conf if is_closed else 1 - conf
        cls = behaviour[t]
        if rng.random() < dist_err * (2 if burst else 1):
            cls = rng.integers(0, N_DIST)
        seq[t, 1:] = _softmax_like(rng, cls, N_DIST, rng.uniform(0.4, 0.99))
    return seq

def simulate_dataset(n, T, seed):
    rng = np.random.default_rng(seed)
    y = np.arange(n) % 3; rng.shuffle(y)
    return np.stack([simulate_sequence(rng, T, int(l), eye_err, dist_err) for l in y]), y.astype(np.int64)


class TemporalAttention(nn.Module):
    def __init__(self, hidden):
        super().__init__(); self.proj = nn.Linear(hidden, hidden); self.score = nn.Linear(hidden, 1, bias=False)
    def forward(self, out, mask=None):
        s = self.score(torch.tanh(self.proj(out))).squeeze(-1)
        if mask is not None: s = s.masked_fill(~mask, float("-inf"))
        w = torch.softmax(s, 1)
        return torch.bmm(w.unsqueeze(1), out).squeeze(1), w

class TemporalAttentionLSTM(nn.Module):
    def __init__(self, input_dim, num_classes, hidden_size=64, num_layers=1, dropout=0.2, bidirectional=False):
        super().__init__()
        self.input_norm = nn.LayerNorm(input_dim)
        self.lstm = nn.LSTM(input_dim, hidden_size, num_layers=num_layers, batch_first=True,
                            dropout=dropout if num_layers > 1 else 0.0, bidirectional=bidirectional)
        d = hidden_size * (2 if bidirectional else 1)
        self.attention = TemporalAttention(d); self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(d, num_classes))
        self.config = dict(input_dim=input_dim, num_classes=num_classes, hidden_size=hidden_size,
                           num_layers=num_layers, dropout=dropout, bidirectional=bidirectional)
    def forward(self, x, return_attention=False):
        out, _ = self.lstm(self.input_norm(x)); ctx, w = self.attention(out); logits = self.head(ctx)
        return (logits, w) if return_attention else logits

tc = CFG["temporal"]; T = tc["seq_len"]
t0 = time.time()
Xtr, ytr = simulate_dataset(tc["n_train"], T, CFG["seed"])
Xva, yva = simulate_dataset(tc["n_val"], T, CFG["seed"] + 1)
Xte, yte = simulate_dataset(tc["n_test"], T, CFG["seed"] + 2)
print(f"Simulated {len(Xtr)}/{len(Xva)}/{len(Xte)} train/val/test sequences of shape {Xtr.shape[1:]} in {time.time()-t0:.0f}s")

In [ ]:
set_seed(CFG["seed"])
temporal = TemporalAttentionLSTM(1 + N_DIST, 3, tc["hidden_size"], tc["num_layers"], tc["dropout"], tc["bidirectional"]).to(DEVICE)
t_opt = torch.optim.AdamW(temporal.parameters(), lr=tc["lr"], weight_decay=tc["weight_decay"])
t_sched = torch.optim.lr_scheduler.ReduceLROnPlateau(t_opt, factor=0.5, patience=2)
tl = lambda X, y, bs, sh: DataLoader(TensorDataset(torch.from_numpy(X), torch.from_numpy(y)), batch_size=bs, shuffle=sh)
t_fit = fit("temporal_lstm_attention", temporal, tl(Xtr, ytr, tc["batch_size"], True), tl(Xva, yva, 512, False),
            nn.CrossEntropyLoss(), t_opt, t_sched, tc["epochs"], tc["early_stopping_patience"],
            {"model_config": temporal.config, "class_names": TEMPORAL_CLASSES, "seq_len": T,
             "training_data": "SIMULATED encoder-output sequences (not real video)"})

temporal.eval()
with torch.no_grad():
    logits, attn = temporal(torch.from_numpy(Xte).to(DEVICE), return_attention=True)
t_prob = torch.softmax(logits.float(), 1).cpu().numpy(); t_pred = t_prob.argmax(1); attn = attn.cpu().numpy()
t_metrics = compute_metrics(yte, t_pred, t_prob, TEMPORAL_CLASSES)
t_report = classification_report(yte, t_pred, target_names=TEMPORAL_CLASSES, digits=4, zero_division=0)
print("=== Temporal head on SIMULATED test sequences (not real-world performance) ===\n" + t_report)
plot_confusion(confusion_matrix(yte, t_pred, labels=[0, 1, 2]), TEMPORAL_CLASSES, "temporal_simulated_test",
               "Temporal head (SIMULATED test)")
(DIRS["reports"] / "temporal_simulated_test_metrics.json").write_text(json.dumps(
    {**t_metrics, "data": "simulated", "warning": "Not real-world drowsiness performance"}, indent=2))
(DIRS["reports"] / "temporal_simulated_test_report.txt").write_text(t_report)

fig, axes = plt.subplots(1, 2, figsize=(15, 3.2))
for ax, cls, sig_name in ((axes[0], 2, "p(eye closed)"), (axes[1], 1, "1 - p(safe driving)")):
    hits = np.flatnonzero((yte == cls) & (t_pred == cls))
    if len(hits) == 0:
        ax.set_title(f"no correctly classified {TEMPORAL_CLASSES[cls]} sequence"); continue
    i = int(hits[0]); sig = Xte[i, :, 0] if cls == 2 else 1 - Xte[i, :, 1]
    ax.bar(np.arange(T), attn[i], color="tab:orange", alpha=.7, label="attention")
    ax2 = ax.twinx(); ax2.plot(sig, color="tab:blue"); ax2.set_ylim(0, 1); ax2.set_ylabel(sig_name)
    ax.set_title(f"Attention over a simulated {TEMPORAL_CLASSES[cls]} window"); ax.set_xlabel("frame")
plt.tight_layout(); plt.savefig(DIRS["plots"] / "temporal_attention_examples.png", dpi=120); plt.show()
print("Note: a unidirectional LSTM's later states summarise more history, so attention also reflects recency.")

export(temporal, DIRS["models"] / "temporal_lstm_attention.pth",
       {"model_config": temporal.config, "class_names": TEMPORAL_CLASSES, "seq_len": T, "training_data": "simulated"})
RESULTS_SUMMARY["temporal"] = {"data": "SIMULATED", **{k: v for k, v in t_fit.items() if k != "history"},
                               **{k: v for k, v in t_metrics.items() if k != "per_class"}}

### Multi-frame decision ablation (simulated)

This compares the project's decision strategies on fresh simulated windows:

* **single frame:** the last frame only, no smoothing;
* **rules only:** PERCLOS-style closed-eye ratio, longest closure, and unsafe-behaviour ratio;
* **LSTM only;**
* **blended** (the app default): `score = (1−w)·rule + w·p_lstm`, firing when `score ≥ 1 − w/2`, with w = 0.5.

CRITICAL is counted as DROWSY because the simulator has no CRITICAL ground truth. Again, this is evidence about the mechanism, not real-world performance.

In [ ]:
def decide(frames_pc, frames_dist, c, tprobs=None):
    # Project decision engine on one window. frames_pc: p(closed) or nan; frames_dist: [T,10].
    pc = np.asarray(frames_pc, dtype=float); n = len(pc)
    known = ~np.isnan(pc); n_known = int(known.sum())
    closed = known & (np.nan_to_num(pc) >= c["eye_closed_conf"])
    closed_ratio = closed.sum() / n_known if n_known >= max(1, c["min_frames"] // 2) else None
    run = best = 0
    for v in closed:
        run = run + 1 if v else 0; best = max(best, run)
    rule_drowsy = 0.0 if closed_ratio is None else min(1.0, max(closed_ratio / c["drowsy_closed_ratio"],
                                                                 best / c["drowsy_consecutive_closed"]))
    idx = frames_dist.argmax(1); conf = frames_dist.max(1)
    unsafe = (idx != 0) & (conf >= c["distraction_conf"])
    rule_dist = min(1.0, unsafe.mean() / c["distraction_ratio"])
    w = c["temporal_model_weight"] if tprobs is not None else 0.0
    ds = (1 - w) * rule_drowsy + w * (tprobs[2] if tprobs is not None else 0)
    dd = (1 - w) * rule_dist + w * (tprobs[1] if tprobs is not None else 0)
    cut = 1 - w / 2
    drowsy = ds >= cut and closed_ratio is not None; dist = dd >= cut
    extreme = closed_ratio is not None and (closed_ratio >= c["critical_ratio"] or best >= 2 * c["drowsy_consecutive_closed"])
    if drowsy and (dist or extreme): return "CRITICAL"
    return "DROWSY" if drowsy else "DISTRACTED" if dist else "SAFE"

STATE_TO_IDX = {"SAFE": 0, "DISTRACTED": 1, "DROWSY": 2, "CRITICAL": 2}
dcfg = CFG["decision"]
Xab, yab = simulate_dataset(1500 if not QUICK_RUN else 300, T, CFG["seed"] + 99)
with torch.no_grad():
    p_lstm = torch.softmax(temporal(torch.from_numpy(Xab).to(DEVICE)).float(), 1).cpu().numpy()
single_cfg = {**dcfg, "min_frames": 1, "drowsy_closed_ratio": 1.0, "drowsy_consecutive_closed": 1, "distraction_ratio": 1.0}
preds = {"single_frame": [], "rules_only": [], "lstm_only": list(p_lstm.argmax(1)), "blended": []}
for i in range(len(Xab)):
    pc = np.where(np.abs(Xab[i, :, 0] - 0.5) < 1e-6, np.nan, Xab[i, :, 0]); dist = Xab[i, :, 1:]
    preds["single_frame"].append(STATE_TO_IDX[decide(pc[-1:], dist[-1:], single_cfg)])
    preds["rules_only"].append(STATE_TO_IDX[decide(pc, dist, dcfg)])
    preds["blended"].append(STATE_TO_IDX[decide(pc, dist, dcfg, p_lstm[i])])
abl = []
for k, p in preds.items():
    p = np.asarray(p); m = compute_metrics(yab, p, None, TEMPORAL_CLASSES)
    abl.append({"strategy": k, "accuracy": m["accuracy"], "macro_f1": m["macro_f1"],
                "false_alarms_on_SAFE": float((p[yab == 0] != 0).mean()),
                **{f"recall_{c}": m["per_class"][c]["recall"] for c in TEMPORAL_CLASSES}})
ablation = pd.DataFrame(abl)
ablation.to_csv(DIRS["reports"] / "temporal_decision_ablation_SIMULATED.csv", index=False)
print("Decision-strategy ablation on SIMULATED windows:")
print(ablation.round(4).to_string(index=False))
RESULTS_SUMMARY["temporal"]["ablation"] = abl

## 13. Checkpoints and reproducibility information

* `checkpoints/<run>_last.pth`: full training state after the last epoch, for resuming. It is kept in `/kaggle/working` but left out of the ZIP to keep the download small. To resume in a later session, save a notebook version (so `/kaggle/working` becomes its output) and attach that output as an input.
* `checkpoints/<run>_best.pth`: best-validation weights per run.
* `models/`: the selected models in the DriveAwareAI app format. Copy these into the project's `models/` folder to use them in the Streamlit dashboard.
* `config.json`, `reproducibility.json`: configuration, seeds, package versions, GPU and data splits.

In [ ]:
for p in sorted(DIRS["models"].glob("*.pth")):
    ck = torch.load(p, map_location="cpu", weights_only=False)
    print(f"{p.name:32s} {p.stat().st_size / 1e6:6.1f} MB | keys: {[k for k in ck if k != 'model_state_dict']}")

# verify the exported image checkpoints reload into the architecture and reproduce test predictions
chk = build_resnet_classifier(10, dc["dropout"], pretrained=False).to(DEVICE)
chk.load_state_dict(torch.load(DIRS["models"] / "distraction_resnet18.pth", map_location=DEVICE, weights_only=False)["model_state_dict"])
xb = torch.stack([best_d["test_ds"][i][0] for i in range(min(8, len(best_d["test_ds"])))]).to(DEVICE)
chk.eval(); best_d["model"].eval()
with torch.no_grad():
    assert torch.equal(chk(xb).argmax(1), best_d["model"](xb).argmax(1)), "exported checkpoint does not reproduce predictions"
print("Exported distraction checkpoint reloads and reproduces predictions.")

repro = {"seed": CFG["seed"], "quick_run": QUICK_RUN, "device": str(DEVICE), "amp": USE_AMP,
         "gpus": [torch.cuda.get_device_name(i) for i in range(N_GPUS)], "versions": versions,
         "torch_cuda": torch.version.cuda, "cudnn": torch.backends.cudnn.version() if DEVICE.type == "cuda" else None,
         "datasets": {k: str(v) for k, v in PATHS.items()}, "splits": SPLITS,
         "selected_distraction_config": best_mode, "notes": [
             "State Farm split by driver (GroupShuffleSplit); test split used only for final evaluation.",
             "No horizontal flip on State Farm (would swap left/right classes).",
             "Temporal model trained/tested on SIMULATED sequences; metrics are not real-world performance.",
             "cudnn.benchmark=True: results are reproducible up to GPU non-determinism."]}
(RESULTS / "reproducibility.json").write_text(json.dumps(repro, indent=2, default=str))

## 14. Final results summary and download

All metrics below are computed in this run from real predictions. Distraction and eye-state numbers are on held-out test data. Temporal numbers are on **simulated** data, as labelled.

In [ ]:
lines = ["# DriveAwareAI: Kaggle run results", "",
         f"Device: {DEVICE} {repro['gpus']} | QUICK_RUN: {QUICK_RUN} | seed {CFG['seed']}", "",
         "## Image models (real held-out test data)", "",
         "```", comparison.round(4).to_string(index=False), "```",
         "", f"Selected distraction configuration (lowest validation loss): **{best_mode}**", "",
         "## Temporal head (SIMULATED sequences: not real-world drowsiness performance)", "",
         "```", ablation.round(4).to_string(index=False), "```", "",
         "## Files", ""]
for p in sorted(RESULTS.rglob("*")):
    if p.is_file():
        lines.append(f"- {p.relative_to(RESULTS)} ({p.stat().st_size / 1e6:.2f} MB)")
(RESULTS / "SUMMARY.md").write_text("\n".join(lines))
(RESULTS / "results_summary.json").write_text(json.dumps(RESULTS_SUMMARY, indent=2, default=str))

print(f"Distraction ({best_mode}) test: acc {dm['accuracy']:.4f} | macro P {dm['macro_precision']:.4f} | "
      f"macro R {dm['macro_recall']:.4f} | macro F1 {dm['macro_f1']:.4f}")
print(f"Eye state test:           acc {eye_metrics['accuracy']:.4f} | macro P {eye_metrics['macro_precision']:.4f} | "
      f"macro R {eye_metrics['macro_recall']:.4f} | macro F1 {eye_metrics['macro_f1']:.4f}")
print(f"Temporal (SIMULATED):     acc {t_metrics['accuracy']:.4f} | macro F1 {t_metrics['macro_f1']:.4f}")

In [ ]:
zip_path = WORKING / "DriveAwareAI_results.zip"
if zip_path.exists():
    zip_path.unlink()
# The *_last.pth resume states (weights + optimizer, ~3x model size) stay in /kaggle/working only,
# to keep the download small. Everything else, including the per-run best checkpoints, is zipped.
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(RESULTS.rglob("*")):
        if p.is_file() and not p.name.endswith("_last.pth"):
            zf.write(p, p.relative_to(WORKING))
with zipfile.ZipFile(zip_path) as zf:
    bad = zf.testzip()
    if bad is not None:
        raise RuntimeError(f"ZIP verification failed at {bad}")
    n_files = len(zf.namelist())
print(f"Created {zip_path} ({zip_path.stat().st_size / 1e6:.1f} MB, {n_files} files, verified)")

try:
    from IPython.display import FileLink, display
    os.chdir(WORKING)                 # FileLink paths are relative to the notebook's working dir
    display(FileLink(zip_path.name, result_html_prefix="⬇️ Download results: "))
except ImportError:
    print("Download from the Output panel:", zip_path)